In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# External imports
import logging
import warnings
from pathlib import Path

# from utils.masks import create_circular_mask
# from utils.metrics import calculate_L2Norm_error
import qim3d
from cil.io import TIFFWriter
from cil.optimisation.algorithms import FISTA, SIRT
from cil.optimisation.functions import IndicatorBox, L1Norm, LeastSquares
from cil.optimisation.utilities import callbacks
from cil.optimisation.utilities.callbacks import (
    EarlyStoppingObjectiveValue,
    ProgressCallback,
)
from cil.plugins.astra.operators import ProjectionOperator
from cil.plugins.ccpi_regularisation.functions import FGP_TV

# from cil.plugins.tigre import ProjectionOperator
from cil.processors import Binner, Slicer, TransmissionAbsorptionConverter
from cil.recon import FBP, FDK
from cil.utilities.display import show1D, show2D, show_geometry
from gvxrPython3.cil.JSON2gVXRDataReader import JSON2gVXRDataReader
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity as ssim

import common

In [ ]:
data_dir = common.data_dir
print(f"{data_dir=}")

case_dir=PosixPath('/zhome/1a/a/219376/Code/gvxr-cil-qim3d-workshop/integrated_case')
data_dir=PosixPath('/zhome/1a/a/219376/Code/gvxr-cil-qim3d-workshop/integrated_case/data')


In [3]:
data = JSON2gVXRDataReader(str(data_dir / "projections" / "simulation.json")).read()


[512, 512]
[0.020000000298023225, 0.020000000298023225]


In [4]:
# Because we get full attentuation for some parts of the object,
# we need to handle the case of zero photon count

arr = data.as_array()
min_val = arr[arr > 0].min()
data = TransmissionAbsorptionConverter(min_intensity=min_val)(data)
data.reorder(order="astra")

In [5]:
slice_half = (None, None, 2)
roi_binning = {"horizontal": slice_half, "vertical": slice_half}
roi_slice = {"angle": slice_half}

binner = Binner(roi_binning)
binner.set_input(data)
data_binned = binner.get_output()

slicer = Slicer(roi_slice)
slicer.set_input(data_binned)
data_reduced = slicer.get_output()


In [6]:
ig_reduced = data_reduced.geometry.get_ImageGeometry()

A = ProjectionOperator(ig_reduced, data_reduced.geometry)

b = data_reduced
F = LeastSquares(A, b)

alpha = 1e-4
G = alpha * L1Norm()

initial = ig_reduced.allocate(0.0)
fista = FISTA(f=F, g=G, initial=initial, update_objective_interval=10)


In [7]:
trial_alphas = [
    1e-3,
    1e-2,
    1e-1,
]

fista_runs = {}

for alpha in trial_alphas:
    A = ProjectionOperator(ig_reduced, data_reduced.geometry)

    b = data_reduced
    F = LeastSquares(A, b)

    G = alpha * FGP_TV(device="gpu")

    initial = ig_reduced.allocate(0.0)
    fista = FISTA(f=F, g=G, initial=initial, update_objective_interval=10)

    fista_runs[alpha] = fista

/tmp/ipykernel_607208/1424104442.py:15: UserWarning: Note that the default behaviour now sets the nonnegativity constraint to False 
  G = alpha * FGP_TV(device="gpu")


In [8]:
for alpha, fista in fista_runs.items():
    print(f"Running FISTA with alpha={alpha}")
    fista.run(250)

Running FISTA with alpha=0.001


  0%|          | 0/250 [00:00<?, ?it/s]

Running FISTA with alpha=0.01


  0%|          | 0/250 [00:00<?, ?it/s]

Running FISTA with alpha=0.1


  0%|          | 0/250 [00:00<?, ?it/s]

In [11]:
qim3d.viz.planes(list(fista_runs.values())[2].solution.as_array())

FigureWidget({
    'data': [{'cmax': 255,
              'cmin': 0,
              'colorscale': [[0.0, 'rgb(0, 0, 4)'], [0.00392156862745098, 'rgb(1,
                             0, 5)'], [0.00784313725490196, 'rgb(1, 1, 6)'], ...,
                             [0.9921568627450981, 'rgb(252, 249, 187)'],
                             [0.996078431372549, 'rgb(252, 251, 189)'], [1.0,
                             'rgb(252, 253, 191)']],
              'lighting': {'ambient': 1, 'diffuse': 0, 'fresnel': 0, 'roughness': 1, 'specular': 0},
              'name': 'Z',
              'opacity': 1.0,
              'showlegend': False,
              'showscale': False,
              'surfacecolor': array([[17, 17, 17, ..., 17, 17, 17],
                                     [17, 17, 17, ..., 17, 17, 17],
                                     [17, 17, 17, ..., 17, 17, 17],
                                     ...,
                                     [17, 17, 17, ..., 16, 16, 16],
                        

In [13]:
for alpha, fista in fista_runs.items():
    writer = TIFFWriter(
        fista.solution,
        file_name=data_dir / f"reconstruction_fista__alpha_{alpha}" / "slice",
    )
    writer.write()